# 02 — Exploratory analysis

## Pertanyaan

Bagaimana tren indikator nasional dan moneter, pertumbuhan GDP per kapita dan populasi, kandidat outlier, pola bulan kalender, serta posisi pertumbuhan GDP Indonesia di ASEAN?

## Metode

Analisis memakai perubahan antarperiode, aturan IQR 1,5 kali rentang antarkuartil untuk kandidat outlier, rata-rata menurut bulan kalender untuk eksplorasi seasonality, dan snapshot ASEAN pada tahun terbaru yang tersedia.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from analytics.descriptive.data_access import DATASET_SOURCES
from analytics.descriptive.notebook_support import insight, prepare_notebook, save_figure
from analytics.descriptive.statistics import add_growth_rates, iqr_outliers, latest_asean_snapshot, latest_indonesia_vs_asean, monthly_seasonality

data = prepare_notebook()
national = data.national.copy()
monetary = data.monetary.copy()
for column in ["gdp_growth_percent", "inflation_percent", "unemployment_percent", "population", "gdp_per_capita_current_usd"]:
    national[column] = pd.to_numeric(national[column], errors="coerce")
for column in ["bi_rate_percent", "jisdor_idr_per_usd", "jisdor_mom_percent_change"]:
    monetary[column] = pd.to_numeric(monetary[column], errors="coerce")

## Hasil

In [ ]:
national_growth = add_growth_rates(
    national, ["population", "gdp_per_capita_current_usd"]
)
outliers = iqr_outliers(
    national, ["gdp_growth_percent", "inflation_percent", "unemployment_percent"]
)
jisdor_change_by_month = monthly_seasonality(monetary, "jisdor_mom_percent_change")
display(national_growth.tail(10))
display(outliers)
display(jisdor_change_by_month)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 12))
axes[0].plot(national["observation_date"], national["gdp_growth_percent"], marker="o", label="GDP growth")
axes[0].plot(national["observation_date"], national["inflation_percent"], marker="o", label="Inflasi")
axes[0].set(title="Tren indikator nasional", ylabel="Persen")
axes[0].legend()
axes[1].plot(national_growth["observation_date"], national_growth["population_growth_percent"], label="Pertumbuhan populasi")
axes[1].plot(national_growth["observation_date"], national_growth["gdp_per_capita_current_usd_growth_percent"], label="Pertumbuhan GDP per kapita nominal USD")
axes[1].set(title="Perubahan tahunan", ylabel="Persen")
axes[1].legend()
axes[2].plot(monetary["observation_date"], monetary["jisdor_idr_per_usd"], color="tab:green", label="JISDOR")
rate_axis = axes[2].twinx()
rate_axis.plot(monetary["observation_date"], monetary["bi_rate_percent"], color="tab:red", label="BI-Rate")
axes[2].set(title="Kondisi moneter bulanan", ylabel="IDR per USD")
rate_axis.set_ylabel("Persen")
fig.tight_layout()
save_figure(fig, "02_trends_and_growth.png")
display(fig)
plt.close(fig)

In [ ]:
gdp_code = "NY.GDP.MKTP.KD.ZG"
indonesia_vs_asean = latest_indonesia_vs_asean(data.asean)
display(indonesia_vs_asean)
asean_gdp = latest_asean_snapshot(data.asean, gdp_code)
asean_gdp["value"] = pd.to_numeric(asean_gdp["value"], errors="coerce")
colors = ["tab:red" if code == "IDN" else "tab:blue" for code in asean_gdp["region_code"]]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(jisdor_change_by_month["calendar_month"], jisdor_change_by_month["mean"], marker="o")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(title="Rata-rata perubahan JISDOR menurut bulan kalender", xlabel="Bulan", ylabel="Perubahan MoM (%)", xticks=range(1, 13))
axes[1].bar(asean_gdp["region_code"], asean_gdp["value"], color=colors)
axes[1].set(title=f"Pertumbuhan GDP ASEAN ({int(asean_gdp['observation_year'].max())})", ylabel="Persen")
fig.tight_layout()
save_figure(fig, "02_seasonality_and_asean.png")
display(fig)
plt.close(fig)

In [ ]:
latest_national = national.sort_values("observation_date").iloc[-1]
display(insight(
    f"Pada observasi nasional terbaru, pertumbuhan GDP adalah {latest_national['gdp_growth_percent']:.2f}% dan inflasi {latest_national['inflation_percent']:.2f}%. Aturan IQR menandai {len(outliers)} kandidat outlier pada tiga indikator persentase.",
    frame=national,
    source=DATASET_SOURCES["national"],
    limitation="Outlier statistik adalah kandidat untuk diperiksa, bukan otomatis kesalahan data atau anomali ekonomi. Pertumbuhan GDP per kapita memakai nilai nominal USD sehingga juga dipengaruhi harga dan kurs.",
))
indonesia_row = asean_gdp.loc[asean_gdp["region_code"].eq("IDN")].iloc[0]
display(insight(
    f"Indonesia berada pada peringkat nilai pertumbuhan GDP ke-{int(indonesia_row['value_rank_desc'])} dari {int(indonesia_row['country_coverage'])} negara yang memiliki observasi pada snapshot terbaru.",
    frame=asean_gdp,
    source=DATASET_SOURCES["asean"],
    limitation="Peringkat hanya mengurutkan nilai pada satu tahun dan coverage dapat berbeda antarindikator; peringkat tinggi tidak selalu berarti kualitas ekonomi lebih baik.",
))
peak_month = jisdor_change_by_month.loc[jisdor_change_by_month["mean"].idxmax()]
display(insight(
    f"Rata-rata perubahan bulanan JISDOR tertinggi dalam pengelompokan bulan kalender muncul pada bulan ke-{int(peak_month['calendar_month'])}, sebesar {peak_month['mean']:.2f}%.",
    frame=monetary,
    source=DATASET_SOURCES["monetary"],
    limitation="Rata-rata bulan kalender bersifat eksploratif, memakai jumlah tahun terbatas, dan belum membuktikan pola musiman yang stabil.",
))

## Interpretasi

Grafik membantu menemukan perubahan besar, pola waktu, dan observasi yang perlu ditelusuri kembali ke sumber. Perbandingan ASEAN memakai coverage aktual pada periode yang sama.

## Keterbatasan

Analisis ini deskriptif. Perbedaan frekuensi tahunan dan bulanan, definisi indikator, revisi sumber, tren jangka panjang, dan jumlah observasi membatasi perbandingan langsung.